In [1]:
# Cell 1 - Install Required Libraries
!pip install dash plotly pandas

print("✅ Libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 42.0 MB/s eta 0:00:00
✅ Libraries installed!


In [2]:
# Cell 2 - Generate Clinical Safety Data
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

data = {
    'patient_id': ['PT' + str(i).zfill(4) for i in range(1, n+1)],
    'age': np.random.randint(18, 75, n),
    'gender': np.random.choice(['Male', 'Female'], n),
    'site': np.random.choice(['Site_A', 'Site_B', 'Site_C', 'Site_D'], n),
    'treatment_group': np.random.choice(['Drug', 'Placebo'], n),
    'ae_count': np.random.randint(0, 10, n),
    'ae_severity': np.random.choice(['Mild', 'Moderate', 'Severe'], n),
    'sae_flag': np.random.choice([0, 1], n, p=[0.85, 0.15]),
    'visit_count': np.random.randint(1, 12, n),
    'dropout': np.random.choice([0, 1], n, p=[0.75, 0.25])
}

df = pd.DataFrame(data)
df.to_csv('clinical_safety_data.csv', index=False)
print("✅ Safety Data Created!")
print(df.shape)

✅ Safety Data Created!
(1000, 10)


In [7]:
# Cell 3 - Clinical Safety Dashboard
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'AE Severity Distribution',
        'Average AE Count by Site',
        'Dropout Rate by Treatment',
        'Age Distribution'
    ),
    specs=[[{'type':'pie'},{'type':'bar'}],
           [{'type':'bar'},{'type':'histogram'}]]
)

severity_counts = df['ae_severity'].value_counts()
fig.add_trace(go.Pie(
    labels=severity_counts.index,
    values=severity_counts.values,
    marker_colors=['green','orange','red']),
    row=1, col=1)

site_ae = df.groupby('site')['ae_count'].mean().reset_index()
fig.add_trace(go.Bar(
    x=site_ae['site'],
    y=site_ae['ae_count'],
    marker_color='crimson'),
    row=1, col=2)

dropout = df.groupby(
    'treatment_group')['dropout'].mean().reset_index()
fig.add_trace(go.Bar(
    x=dropout['treatment_group'],
    y=dropout['dropout'],
    marker_color=['steelblue','coral']),
    row=2, col=1)

for sev in df['ae_severity'].unique():
    subset = df[df['ae_severity']==sev]
    fig.add_trace(go.Histogram(
        x=subset['age'],
        name=sev,
        opacity=0.7),
        row=2, col=2)

fig.update_layout(
    height=800,
    title_text="Clinical Trial Safety Monitoring Dashboard",
    title_font_size=20
)

fig.write_html('clinical_safety_dashboard.html')
fig.show()
print("Dashboard Created!")

Dashboard Created!
